# Experiment 4 — when the reasoning is already in the weights

Four math-tuned checkpoints across two generations, against the general models they are meant to
beat, on three difficulty bands, *n* = 100 paired items.

**The claim under test:** when *how to reason* has been trained in, a small specialised model
outperforms a much larger general one. The comparison is `Math-1.5B` against `7B-Instruct`, a
model **4.7× its size**, and it is scored twice: against the general model **suppressed** and
against it **reasoning**. The second is the honest number. If a math model only wins against a
gagged opponent, the claim is much weaker and this notebook has to say so.

**The second finding, which experiment 2 found first:** these models *refuse the blurt
instruction*. Three of four showed 0% compliance at *n* = 20 while every general model complied
100% of the time at a median of 6 tokens. A model that will not stop reasoning when told to is
a model whose reasoning is not coming from the prompt. That is the whole thesis, visible in one
column of table 3.

Four checkpoints rather than two so the generation that shows the message most clearly can be
**chosen from data**. Writes `out/experiment_4/`, with `tables.md` beside `derived.json`.

## How to run this

| models | bands | conditions | items | generations |
|---|---|---|---|---|
| 6 | 3 | 3 | 100 | **5,400** |

Four math-tuned checkpoints and the two general models they are compared against, **all at
bf16**. The 7B rows need about 15 GB of weights each and will not fit a 16 GB card; this is
written to be run somewhere larger, and the section below is the reason that is safe.

Bands are MATH-500 levels 3, 4 and 5 kept **separate**, so the difficulty axis is visible rather
than averaged away. Level 3 holds only 105 problems, so 100 is the ceiling, and every band uses
100 because the bands have to match for a comparison across them to mean anything.

### Running it somewhere faster

This is the first sweep worth moving to a rented GPU, and doing so does not invalidate anything
**provided three things hold**.

1. **Quantisation is pinned by `QUANT_7B`, not by what fits**, and it is `bf16`. Experiment 3
   ran its 7B at 8-bit because 15.24 GB of bf16 weights do not fit a 16 GB card, so its 7B
   numbers are **not comparable** to this run's and must not be mixed with them. Every 7B here,
   math-tuned and general alike, uses one precision, which is the whole reason the general
   comparators are swept in this notebook rather than joined from experiment 3. `QUANT_7B` is
   in the run id, so changing it starts a fresh set rather than resuming an incomparable one.
2. **The Math checkpoints carry a 4,096-token context**, prompt included, against 32k for the
   general models. The sweep clips the cap per model from `model.config` and reports the floor it
   hit; if that floor is low, the hard band is being scored under a budget the general models
   never felt.
3. **Exactly one process.** `harness.TraceStore` has no write lock, and two workers on one
   `run_id` produce duplicate records and double-counted aggregates. The table cells verify
   `len(records) == len(unique keys)` and refuse to build otherwise.

What *will* differ across machines: greedy decoding is deterministic on one machine but not
between them, because floating-point reduction order changes with the kernel and `get_device()`
enables TF32 on CUDA. Expect a handful of items to flip and the shape and ranking to reproduce,
not every digit. The manifest records the device so a divergence has somewhere to be attributed.

On Colab specifically: the working directory is ephemeral, so mount Drive or pull
`out/experiment_4/traces.jsonl` down before the runtime recycles. The run is resumable from that
file alone.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs.
# ---------------------------------------------------------------------------
N_ITEMS    = 100
FEW_SHOT_K = 2
GEN_CAP    = 2048
SEED       = 123

# Three bands, kept separate. L3 holds 105 problems, L4 128, L5 134, so n=100 plus
# two exemplars fits every band and the bands stay the same size as each other.
DATASETS_USED = ["math500_l3", "math500_l4", "math500_l5"]

# No zero_shot_cot. Experiment 3 measured it as a clean null on the hard band even
# for general models, which already chain unprompted; on models trained to reason
# there is less for it to do, not more. forced_blurt stays because it is the
# condition these models REFUSE, and that refusal is this experiment's headline.
CONDITIONS = ["forced_blurt", "bare", "few_shot"]

# Pinned here rather than chosen by what fits the machine. Every 7B in this run uses
# it -- math-tuned and general alike -- so the specialist-versus-generalist comparison
# in table 5 has no precision difference inside it. Changing this changes RUN_ID, so
# nothing incomparable resumes. Experiment 3's 7B ran at 8bit and its numbers are NOT
# comparable to these.
QUANT_7B = "bf16"          # needs ~15 GB per 7B; "8bit" fits 16 GB and re-runs every 7B row

MATH_MODELS = [
    dict(id="Qwen/Qwen2-Math-1.5B-Instruct",   tag="Qwen2-Math-1.5B",   quant="bf16",
         gen="Qwen2",   size="1.5B", kind="math"),
    dict(id="Qwen/Qwen2.5-Math-1.5B-Instruct", tag="Qwen2.5-Math-1.5B", quant="bf16",
         gen="Qwen2.5", size="1.5B", kind="math"),
    dict(id="Qwen/Qwen2-Math-7B-Instruct",     tag="Qwen2-Math-7B",     quant=QUANT_7B,
         gen="Qwen2",   size="7B",   kind="math"),
    dict(id="Qwen/Qwen2.5-Math-7B-Instruct",   tag="Qwen2.5-Math-7B",   quant=QUANT_7B,
         gen="Qwen2.5", size="7B",   kind="math"),
]

# The comparison targets, swept HERE rather than joined from experiment 3: same
# items, same run, same quantisation, so the headline contrast is paired at the
# problem level instead of being two runs laid beside each other. This also fills
# the level 4-5 gap Qwen2-7B has never been measured on.
INCLUDE_COMPARATORS = True
GENERAL_MODELS = [
    dict(id="Qwen/Qwen2-7B-Instruct",   tag="Qwen2-7B",   quant=QUANT_7B,
         gen="Qwen2",   size="7B", kind="general"),
    dict(id="Qwen/Qwen2.5-7B-Instruct", tag="Qwen2.5-7B", quant=QUANT_7B,
         gen="Qwen2.5", size="7B", kind="general"),
]

ROSTER = MATH_MODELS + (GENERAL_MODELS if INCLUDE_COMPARATORS else [])

# The pairings table 5 is built from: a specialist against the generalist of the same
# generation. Stated as data so the comparison cannot drift from what the prose says.
PAIRINGS = [dict(small="Qwen2-Math-1.5B",   large="Qwen2-7B",   gen="Qwen2",   ratio=4.7),
            dict(small="Qwen2.5-Math-1.5B", large="Qwen2.5-7B", gen="Qwen2.5", ratio=4.7)]

PLANNED = len(ROSTER) * len(DATASETS_USED) * len(CONDITIONS) * N_ITEMS
print(f"{len(ROSTER)} models x {len(DATASETS_USED)} bands x {len(CONDITIONS)} conditions "
      f"x {N_ITEMS} items = {PLANNED:,} generations")
print(f"7B precision: {QUANT_7B}"
      + ("" if INCLUDE_COMPARATORS else "   (comparators OFF: table 5 will not build)"))
if QUANT_7B != "bf16":
    print("   NOTE: not full precision. Table 5 stays internally consistent, but the page\n"
          "   must say so beside any number that crosses the 1.5B/7B boundary.")

In [ ]:
import json
import random
import re
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm.auto import tqdm

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/experiment_4"); OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
TABLES_PATH = OUT_DIR / "tables.md"

mean, median, wilson = harness.mean, harness.median, harness.wilson
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## The item sets

One draw per band, from the same seed, so the three bands are independent samples of the same
size and every model sees identical problems. Exemplars come from outside the scored items and
are checked to carry the answer format the stem asks for: a demonstration that disagrees with the
instruction measures a conflict rather than an example.

In [ ]:
def box_exemplar(row):
    """Make the demonstration agree with the instruction: every exemplar ends in
    \\boxed{answer}. MATH-500's reference solutions already do, so this is a no-op
    here and an assertion that it stayed one."""
    sol = row["solution"].strip()
    if "\\boxed" in sol:
        return row
    return {**row, "solution": f"{sol}\nThe final answer is $\\boxed{{{row['answer']}}}$."}


def draw(rows, n, seed_offset):
    rows = sorted(rows, key=lambda r: r["unique_id"])
    random.Random(SEED + seed_offset).shuffle(rows)
    return rows[:n], [box_exemplar(r) for r in rows[n:n + FEW_SHOT_K]]


math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")
BANDS = {f"math500_l{n}": [r for r in math500 if int(r["level"]) == n] for n in (3, 4, 5)}

DATASETS = {}
for name in DATASETS_USED:
    rows = BANDS[name]
    assert len(rows) >= N_ITEMS + FEW_SHOT_K, \
        f"{name} holds {len(rows)} problems, too few for n={N_ITEMS} plus {FEW_SHOT_K} exemplars"
    items, shots = draw(rows, N_ITEMS, 0)
    DATASETS[name] = {"items": items, "exemplars": shots,
                      "hash": harness.sha256_of(r["unique_id"] for r in items)}
    print(f"{name:<13} {len(items)} scored + {len(shots)} exemplars   of {len(rows)} available   "
          f"sha256 {DATASETS[name]['hash'][:12]}")

assert all("\\boxed" in e["solution"] for d in DATASETS.values() for e in d["exemplars"]), \
    "an exemplar lacks the answer format the stem asks for"
# Bands are separate samples of separate problems; nothing should appear in two.
_seen = [r["unique_id"] for d in DATASETS.values() for r in d["items"]]
assert len(_seen) == len(set(_seen)), "an item appears in more than one band"
print("\nall exemplars carry the answer format; no item appears in two bands")

## Grading

Inherited from ch03, unchanged. The preflight fails loudly rather than silently scoring a shape this sweep produces.

In [ ]:
CHECKS = [
    (r"The answer is \boxed{9}.",                    "9",     True),
    (r"27",                                           "9",     False),   # a wrong blurt
    (r"\boxed{\frac{1}{2}}",                         "0.5",   True),
    (r"\boxed{1,024}",                               "1024",  True),
    (r"I am not sure.",                               "9",     False),
    # what a math model does to the blurt instruction: reasons anyway, then answers.
    (r"Let me work through it. 23-20=3, then 3+6=9. So \boxed{9}.", "9", True),
]
bad = [(t, g, e) for t, g, e in CHECKS
       if bool(grade_answer(extract_final_candidate(t), g)) != e]
assert not bad, f"grader behaved unexpectedly on {len(bad)} check(s): {bad}"
print(f"grader preflight: {len(CHECKS)}/{len(CHECKS)} as expected")

## The three conditions

`bare` is the default. `forced_blurt` is the instruction these models are expected to refuse, and
`few_shot` is the demonstration. The suffixes are word-for-word experiment 3's, so the two runs
measure the same conditions and their tables can sit beside each other.

In [ ]:
BLURT_SUFFIX = "\n\nGive only the final answer. No explanation, no working, no other text."

def build_prompt(condition, question, exemplars):
    stem = render_prompt(question)
    if condition == "bare":
        return stem
    if condition == "forced_blurt":
        return stem + BLURT_SUFFIX
    shots = "".join(render_prompt(e["problem"]) + "\n" + e["solution"].strip() + "\n\n"
                    for e in exemplars[:FEW_SHOT_K])
    return shots + stem

# COMPLIES_UNDER is the blurt-compliance threshold and it is the measurement, not a
# formatting nicety: a model that answers in 40 tokens obeyed, one that writes 400
# refused. 40 rather than 32 because a levels 4-5 answer is often not a bare number.
MIN_STEP_CHARS, MIN_CHAIN_STEPS, COMPLIES_UNDER = 12, 2, 40



## The sweep

One model resident at a time. The cap is clipped per model from `model.config`, because the Math
checkpoints carry a 4,096-token context against 32k for the general models, and a few-shot prompt
on a level-5 problem is not small. The floor that clip reached is reported and asserted: if the
hard band ends up scored under a budget the general models never felt, the comparison is measuring
the context window rather than the training.

In [ ]:
RUN_ID = f"exp4-n{N_ITEMS}-k{FEW_SHOT_K}-cap{GEN_CAP}-q{QUANT_7B}-{SEED}"
STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("tag", "dataset", "condition", "problem_id"))
STORE.report()

device = get_device()
DTYPE = harness.pick_dtype(device)
CAP_FLOOR = {}          # tag -> smallest cap any item on that model was given


def sweep_model(entry):
    loaded = harness.load_model(entry["id"], device=device, dtype=DTYPE,
                                load_in_8bit=entry["quant"] == "8bit")
    model, tok = loaded["model"], loaded["tokenizer"]
    ctx = getattr(model.config, "max_position_embeddings", 32768)
    print(f"      context {ctx:,} tokens   quant {entry['quant']}")
    smallest = GEN_CAP

    for ds_name, ds in DATASETS.items():
        for cond in CONDITIONS:
            bar = harness.progress(ds["items"], desc=f"{entry['tag'][:22]:<22} {ds_name:<12} {cond:<13}",
                       unit="item")
            for row in bar:
                if STORE.get(entry["tag"], ds_name, cond, row["unique_id"]) is not None:
                    continue
                text = build_prompt(cond, row["problem"], ds["exemplars"])
                chat = tok.apply_chat_template([{"role": "user", "content": text}],
                                               tokenize=False, add_generation_prompt=True)
                n_prompt = len(tok(chat, add_special_tokens=False).input_ids)
                cap = max(64, min(GEN_CAP, ctx - n_prompt - 32))
                smallest = min(smallest, cap)
                g = harness.complete(chat, model, tok, stop=["\nQuestion:"], max_new_tokens=cap)
                got = extract_final_candidate(g["text"])
                STORE.append({
                    "tag": entry["tag"], "model_id": entry["id"],
                    "quant": entry["quant"], "kind": entry["kind"],
                    "gen": entry["gen"], "size": entry["size"],
                    "dataset": ds_name, "condition": cond,
                    "problem_id": row["unique_id"], "question": row["problem"],
                    "ground_truth": row["answer"], "trace": g["text"], "tokens": g["tokens"],
                    "prompt_tokens": n_prompt, "cap": cap, "context": ctx,
                    "truncated": g["truncated"],
                    "tokens_generated": g["tokens_generated"],
                    "steps": harness.n_steps(g["text"], min_step_chars=MIN_STEP_CHARS),
                    "tokens_to_answer": harness.tokens_to_answer(g["tokens"]),
                    "extracted_answer": got,
                    "correct": bool(grade_answer(got, row["answer"])),
                })
    CAP_FLOOR[entry["tag"]] = smallest
    print(f"      smallest cap given to any item: {smallest:,} tokens")
    del model, loaded
    harness.free_memory()


for entry in ROSTER:
    sweep_model(entry)

squeezed = {t: c for t, c in CAP_FLOOR.items() if c < GEN_CAP}
print("\nsweep complete")
if squeezed:
    print(f"context clipped the cap on: {squeezed}")
    assert min(squeezed.values()) >= 512, (
        "some item was scored under a cap below 512 tokens, which is a budget the general "
        "models never felt. That comparison measures the context window, not the training. "
        "Lower FEW_SHOT_K or drop the hardest band for the affected model.")

## The tables

Four tables in the standard order, plus the one this experiment exists for. Every paired
comparison is an exact McNemar over discordant items only; a single-cell result at *n* ≤ 20 is
noise and does not appear here at all.

In [ ]:
from math import comb

records = [json.loads(l) for l in TRACES_PATH.open(encoding="utf-8") if l.strip()]
records = [r for r in records if r["run_id"] == RUN_ID]

# TraceStore has no write lock. Two processes on one run_id silently produce duplicate
# records and double-counted aggregates; this has happened. Verify before building.
_keys = {(r["tag"], r["dataset"], r["condition"], r["problem_id"]) for r in records}
assert len(records) == len(_keys), (
    f"{len(records)} records but {len(_keys)} unique keys: duplicates in the trace bank. "
    f"Two processes wrote to it. Deduplicate before trusting any number below.")
print(f"{len(records):,} records, {len(_keys):,} unique keys   OK")

B = "\\boxed"
TAGS = [e["tag"] for e in ROSTER]

def by_item(tag, ds, cond):
    return {r["problem_id"]: r for r in records
            if r["tag"] == tag and r["dataset"] == ds and r["condition"] == cond}

def cell(tag, ds, cond):
    rs = list(by_item(tag, ds, cond).values())
    if not rs:
        return None
    k = sum(r["correct"] for r in rs)
    lo, hi = wilson(k, len(rs))
    return {"tag": tag, "dataset": ds, "condition": cond, "n": len(rs),
            "accuracy": k / len(rs), "ci_lo": lo, "ci_hi": hi,
            "chain_share": mean([r["steps"] >= MIN_CHAIN_STEPS for r in rs]),
            "complies": mean([r["tokens_generated"] <= COMPLIES_UNDER for r in rs]),
            "boxed": mean([B in r["trace"] for r in rs]),
            "truncated": mean([r["truncated"] for r in rs]),
            "median_tokens": median([r["tokens_generated"] for r in rs]),
            "median_steps": median([r["steps"] for r in rs])}

TABLE = [c for t in TAGS for ds in DATASETS for cond in CONDITIONS
         if (c := cell(t, ds, cond))]
KIND = {e["tag"]: e["kind"] for e in ROSTER}
LABEL = {"forced_blurt": "forced blurt", "bare": "bare", "few_shot": "+ worked examples"}
MD = []

def get(tag, ds, cond):
    return next((x for x in TABLE if x["tag"] == tag and x["dataset"] == ds
                 and x["condition"] == cond), None)

def emit(title, header, rows, note=None):
    MD.append(f"\n### {title}\n")
    MD.append("| " + " | ".join(header) + " |")
    MD.append("|" + "|".join("---" for _ in header) + "|")
    for r in rows:
        MD.append("| " + " | ".join(str(x) for x in r) + " |")
    if note:
        MD.append(f"\n{note}")
    print(f"\n### {title}")
    w = [max(len(str(r[i])) for r in [header] + rows) for i in range(len(header))]
    print("  " + "  ".join(str(h).ljust(w[i]) for i, h in enumerate(header)))
    for r in rows:
        print("  " + "  ".join(str(c).ljust(w[i]) for i, c in enumerate(r)))
    if note:
        print(f"  {note}")

# --- Table 1: the ladder, per band ---------------------------------------
for ds in DATASETS:
    rows = [[t, LABEL[cond], f"{c['accuracy']:.0%}", f"{c['ci_lo']:.0%}-{c['ci_hi']:.0%}",
             f"{c['median_tokens']:.0f}", f"{c['median_steps']:.0f}", f"{c['chain_share']:.0%}"]
            for t in TAGS for cond in CONDITIONS if (c := get(t, ds, cond))]
    emit(f"Table 1 — the ladder on {ds}",
         ["model", "prompt", "solved", "95% CI", "tokens", "steps", "chain"], rows,
         f"n = {N_ITEMS} paired items per cell.")

In [ ]:
# --- Table 2: what each prompt change is worth ---------------------------
CONTRASTS = [("forced_blurt", "bare", "cost of suppressing reasoning"),
             ("bare", "few_shot", "gain from showing an example")]
rows, SIG = [], []
for ds in DATASETS:
    for t in TAGS:
        for lo_c, hi_c, what in CONTRASTS:
            a, b = by_item(t, ds, hi_c), by_item(t, ds, lo_c)
            if not a or not b:
                continue
            x, y, p = harness.mcnemar(a, b)
            d = (mean([r["correct"] for r in a.values()])
                 - mean([r["correct"] for r in b.values()])) * 100
            rows.append([t, ds, what, f"{d:+.0f}", f"{x}/{y}", f"{p:.3f}",
                         "significant" if p < 0.05 else "not resolved"])
            SIG.append(dict(tag=t, kind=KIND[t], dataset=ds, contrast=what, delta_pts=d,
                            discordant_hi=x, discordant_lo=y, p=p, significant=p < 0.05))
emit("Table 2 — what each prompt change is worth",
     ["model", "band", "change", "delta", "flips +/-", "p", "verdict"], rows,
     "Exact McNemar on paired items. 'flips' counts problems fixed / broken by the change. "
     "For the math models the suppression row is expected to be near zero, which is the point: "
     "an instruction that costs a general model 40 to 70 points costs these nothing.")

# --- Table 3: the refusal, which is this experiment's headline ------------
rows, REFUSAL = [], []
for ds in DATASETS:
    for t in TAGS:
        c = get(t, ds, "forced_blurt")
        ba = get(t, ds, "bare")
        if not (c and ba):
            continue
        rows.append([t, KIND[t], ds, f"{c['complies']:.0%}", f"{c['median_tokens']:.0f}",
                     f"{ba['median_tokens']:.0f}", f"{(c['accuracy'] - ba['accuracy']) * 100:+.0f}",
                     f"{c['boxed']:.0%}"])
        REFUSAL.append(dict(tag=t, kind=KIND[t], dataset=ds, complied=c["complies"],
                            blurt_median_tokens=c["median_tokens"],
                            bare_median_tokens=ba["median_tokens"],
                            accuracy_delta_pts=(c["accuracy"] - ba["accuracy"]) * 100,
                            boxed=c["boxed"]))
emit("Table 3 — who obeys the instruction to stop reasoning",
     ["model", "kind", "band", "complied", "blurt tok", "bare tok", "acc delta", "boxed"], rows,
     f"'complied' is the share answering in <= {COMPLIES_UNDER} tokens. A model near 0% here "
     f"kept reasoning through an explicit instruction not to, and its accuracy delta is near "
     f"zero because nothing was actually suppressed.")

# --- Table 4: headroom ---------------------------------------------------
rows = []
for t in TAGS:
    for ds in DATASETS:
        ba, fs, fb = get(t, ds, "bare"), get(t, ds, "few_shot"), get(t, ds, "forced_blurt")
        if ba and fs and fb:
            rows.append([t, ds, f"{ba['accuracy']:.0%}", f"{100 - ba['accuracy'] * 100:.0f} pts",
                         f"{(fs['accuracy'] - ba['accuracy']) * 100:+.0f}",
                         f"{(ba['accuracy'] - fb['accuracy']) * 100:+.0f}"])
emit("Table 4 — headroom, and what it buys",
     ["model", "band", "bare", "room above bare", "few-shot gain", "suppression cost"], rows,
     "The few-shot gain correlates -0.79 with bare accuracy: prompting cannot move a model "
     "already near the ceiling, and these models are near it on level 3.")

## Table 5 — the comparison this experiment exists for

A 1.5B specialist against a 7B generalist of the same generation, on the same problems, scored
twice. **Suppressed** is the generalist under the blurt instruction; **reasoning** is the
generalist left alone. Only the second is an honest claim about the specialist, and if the
specialist wins only the first, the page has to say so in those words.

In [ ]:
COMPARISON = []
rows = []
if INCLUDE_COMPARATORS:
    for pair in PAIRINGS:
        for ds in DATASETS:
            small = by_item(pair["small"], ds, "bare")
            if not small:
                continue
            for opp_cond, opp_label in (("forced_blurt", "suppressed"), ("bare", "reasoning")):
                large = by_item(pair["large"], ds, opp_cond)
                if not large:
                    continue
                x, y, p = harness.mcnemar(small, large)
                sa = mean([r["correct"] for r in small.values()])
                la = mean([r["correct"] for r in large.values()])
                rows.append([pair["gen"], ds, f"{pair['small']} vs {pair['large']}", opp_label,
                             f"{sa:.0%}", f"{la:.0%}", f"{(sa - la) * 100:+.0f}",
                             f"{x}/{y}", f"{p:.3f}",
                             "specialist wins" if (sa > la and p < 0.05) else
                             ("generalist wins" if (la > sa and p < 0.05) else "not resolved")])
                COMPARISON.append(dict(generation=pair["gen"], dataset=ds,
                                       small=pair["small"], large=pair["large"],
                                       size_ratio=pair["ratio"], opponent=opp_label,
                                       small_accuracy=sa, large_accuracy=la,
                                       delta_pts=(sa - la) * 100,
                                       discordant_small=x, discordant_large=y, p=p,
                                       significant=p < 0.05))
    emit("Table 5 — a 4.7x smaller specialist against a general model",
         ["gen", "band", "matchup", "opponent", "small", "large", "delta",
          "flips +/-", "p", "verdict"], rows,
         "Both models see identical problems, so this is paired. Read the 'reasoning' rows: "
         "a specialist that beats only a gagged generalist has not shown what this page claims.")

    honest = [c for c in COMPARISON if c["opponent"] == "reasoning"]
    won = [c for c in honest if c["significant"] and c["delta_pts"] > 0]
    print(f"\nagainst a generalist that was allowed to reason: specialist wins "
          f"{len(won)} of {len(honest)} matchups at p<0.05")
    if honest and not won:
        print("  The claim in the header does NOT hold on these bands. That is a result, not a")
        print("  failed run -- but the page must not make the size claim, and section 5b of the")
        print("  handoff says so before the numbers were in.")
else:
    print("INCLUDE_COMPARATORS is False, so table 5 cannot be built from this run alone.")

In [ ]:
import matplotlib.pyplot as plt

INK = {"forced_blurt": "#b23", "bare": "#9aa0ab", "few_shot": "#a2704a"}
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4.6 * len(DATASETS), 4.6), sharey=True)
axes = [axes] if len(DATASETS) == 1 else list(axes)
for ax, ds in zip(axes, DATASETS):
    w = 0.8 / len(CONDITIONS)
    for j, cond in enumerate(CONDITIONS):
        # A cell that was not run leaves a GAP, never a zero bar: a reader takes a
        # zero bar for a score of nought.
        found = [(i + j * w - 0.4, c) for i, t in enumerate(TAGS) if (c := get(t, ds, cond))]
        if not found:
            continue
        xs = [p for p, _ in found]
        cs = [c for _, c in found]
        ax.bar(xs, [c["accuracy"] * 100 for c in cs], width=w,
               color=INK[cond], label=LABEL[cond] if ax is axes[0] else None)
        ax.errorbar(xs, [c["accuracy"] * 100 for c in cs],
                    yerr=[[(c["accuracy"] - c["ci_lo"]) * 100 for c in cs],
                          [(c["ci_hi"] - c["accuracy"]) * 100 for c in cs]],
                    fmt="none", ecolor="#41474f", elinewidth=1, capsize=2)
    ax.set_xticks(range(len(TAGS)))
    ax.set_xticklabels(TAGS, rotation=30, ha="right", fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_title(ds, fontsize=9, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("solved (%)")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle(f"math-tuned against general, n={N_ITEMS} paired items, 95% Wilson",
             fontsize=10, x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_experiment_4.png", dpi=160, bbox_inches="tight")
plt.show()

## The record

In [ ]:
derived = {
    "manifest": {
        "notebook": "experiment_4_reasoning_baked_into_weights.ipynb", "run_id": RUN_ID,
        "harness": harness.fingerprint(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "notebook_git_sha": harness.git_sha(),
        "n_items": N_ITEMS, "few_shot_k": FEW_SHOT_K, "gen_cap": GEN_CAP, "seed": SEED,
        "quant_7b": QUANT_7B, "conditions": CONDITIONS, "roster": ROSTER,
        "pairings": PAIRINGS, "include_comparators": INCLUDE_COMPARATORS,
        "cap_floor_by_model": CAP_FLOOR,
        "datasets": {k: {"hash": v["hash"], "n": len(v["items"]),
                         "exemplars": [e["unique_id"] for e in v["exemplars"]]}
                     for k, v in DATASETS.items()},
        # Greedy is deterministic on one machine and not between them, so the device
        # is part of the record rather than a footnote.
        "versions": {"torch": torch.__version__,
                     "transformers": __import__("transformers").__version__,
                     "device": str(device),
                     "gpu": (torch.cuda.get_device_name(0)
                             if torch.cuda.is_available() else None)},
    },
    "grid": TABLE,
    "contrasts": SIG,
    "refusal": REFUSAL,
    "comparison": COMPARISON,
    "thresholds": {"complies_under_tokens": COMPLIES_UNDER, "min_chain_steps": MIN_CHAIN_STEPS},
}
DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
TABLES_PATH.write_text(
    f"# Experiment 4 — when the reasoning is already in the weights\n\n"
    f"`{RUN_ID}` · n={N_ITEMS} paired items · 7B precision {QUANT_7B} · "
    f"{datetime.now(timezone.utc).isoformat(timespec='minutes')}\n"
    + "\n".join(MD) + "\n", encoding="utf-8")

print(f"derived.json   {harness.kb(DERIVED_PATH):>9.1f} KB")
print(f"tables.md      {harness.kb(TABLES_PATH):>9.1f} KB   <- paste-ready")
print(f"traces.jsonl   {harness.kb(TRACES_PATH):>9.1f} KB")
print(f"\n{sum(1 for s in SIG if s['significant'])} of {len(SIG)} prompt contrasts resolved "
      f"at p<0.05; {sum(1 for c in COMPARISON if c['significant'])} of {len(COMPARISON)} "
      f"cross-model matchups")

## Exporting the results

Everything this notebook produced, in one file. A reader is not expected to run this ablation —
it needs more GPU than the page's own notebook does — but they are expected to be able to **see
what it found**, so the export carries the tables, the aggregates and the full trace bank rather
than a screenshot.

`experiment_4.zip` holds:

| file | what it is | who reads it |
|---|---|---|
| `tables.md` | tables 1 to 5, paste-ready markdown | a person |
| `grid.csv` | one row per (model, band, condition) cell | a spreadsheet |
| `contrasts.csv` | one row per paired test, with p and discordant counts | a spreadsheet |
| `comparison.csv` | table 5, the specialist-versus-generalist matchups | a spreadsheet |
| `derived.json` | every aggregate, plus the manifest | the page, and any re-analysis |
| `figure_experiment_4.png` | the reference render | a person |
| `traces.jsonl` | every generation, in full | anyone checking the above |

On Colab the last cell offers the download directly. Elsewhere the zip is simply written beside
the other outputs.

In [ ]:
import csv
import zipfile

def write_csv(path, rows, fields):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)
    return path


GRID_CSV = write_csv(OUT_DIR / "grid.csv", TABLE,
                     ["tag", "dataset", "condition", "n", "accuracy", "ci_lo", "ci_hi",
                      "chain_share", "complies", "boxed", "truncated",
                      "median_tokens", "median_steps"])
CONTRAST_CSV = write_csv(OUT_DIR / "contrasts.csv", SIG,
                         ["tag", "kind", "dataset", "contrast", "delta_pts",
                          "discordant_hi", "discordant_lo", "p", "significant"])
REFUSAL_CSV = write_csv(OUT_DIR / "refusal.csv", REFUSAL,
                        ["tag", "kind", "dataset", "complied", "blurt_median_tokens",
                         "bare_median_tokens", "accuracy_delta_pts", "boxed"])
COMPARISON_CSV = write_csv(OUT_DIR / "comparison.csv", COMPARISON,
                           ["generation", "dataset", "small", "large", "size_ratio", "opponent",
                            "small_accuracy", "large_accuracy", "delta_pts",
                            "discordant_small", "discordant_large", "p", "significant"])

BUNDLE = OUT_DIR / "experiment_4.zip"
members = [TABLES_PATH, DERIVED_PATH, GRID_CSV, CONTRAST_CSV, REFUSAL_CSV, COMPARISON_CSV,
           OUT_DIR / "figure_experiment_4.png", TRACES_PATH]
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for m in members:
        if Path(m).exists():
            z.write(m, arcname=Path(m).name)

print(f"{'file':<32}{'KB':>10}")
for m in members:
    if Path(m).exists():
        print(f"  {Path(m).name:<30}{harness.kb(m):>10,.1f}")
print(f"\n{BUNDLE.name:<32}{harness.kb(BUNDLE):>10,.1f}   <- everything, one file")

# Colab recycles its filesystem, so offer the download rather than leaving it on disk.
try:
    from google.colab import files       # noqa: F401  (absent off Colab, which is the point)
    print("\nColab detected: starting the download.")
    files.download(str(BUNDLE))
except ImportError:
    print(f"\nNot on Colab. The bundle is at {BUNDLE.resolve()}")

## What this run may and may not claim

- **Supported:** whatever tables 1 to 5 report, on MATH-500 levels 3, 4 and 5, at *n* = 100
  paired items, greedy, with the precision recorded in the manifest.
- **Supported:** that the math-tuned checkpoints do or do not obey an explicit instruction to
  stop reasoning (table 3), which is a fact about the checkpoint and not about maths.
- **Not supported:** any cross-model accuracy comparison against experiment 3, whose 7B ran at a
  different precision unless `QUANT_7B` matches it. Within this run every 7B is identical, which
  is why table 5 is built here rather than joined.
- **Not supported:** any claim that these models are better *reasoners* rather than better at
  this benchmark. Three bands of one dataset in one language is a maths result, and the
  specialisation question — whether their reasoning transfers off maths — is not measured here
  and must not be implied.
- **Not supported:** a size claim resting on the suppressed column of table 5. If the specialist
  beats the generalist only when the generalist is gagged, the honest sentence is that it does
  not beat it.